# Counterfactual Viral Etiology Modeling
### How would survival risk change if viral status were different?

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

import shap
import warnings
warnings.filterwarnings("ignore")


In [76]:
clinical = pd.read_csv("/kaggle/input/tcga-lihc-hepatitis-bc-and-transcriptomics-dataset/TCGA_LIHC_Clinical_Viral.csv")
expression = pd.read_csv("/kaggle/input/tcga-lihc-hepatitis-bc-and-transcriptomics-dataset/TCGA_LIHC_Gene_Expression.csv", index_col=0)

clinical.head()
expression.shape



(60660, 424)

In [85]:
# use days_to_death when available otherwise last follow up
clinical["Survival_Days"] = clinical["days_to_death"].fillna(clinical["days_to_last_followup"])

# remove missing survival days
clinical = clinical[clinical["Survival_Days"].notna()]


In [68]:
clinical_filtered = clinical[clinical["Virus_Status"].isin(["HBV", "HCV"])].copy()

clinical_filtered["Virus_Status"].value_counts()

Virus_Status
HBV    177
HCV     33
Name: count, dtype: int64

In [84]:
# Trim expression column names to first 12 characters
expression.columns = expression.columns.str[:12]

common_patients = list(
    set(clinical_filtered["Patient_ID"]).intersection(expression.columns)
)

print("Common patients:", len(common_patients))


Common patients: 205


In [91]:
# Transpose gene matrix
expression_final = expression[common_patients]
X = expression_final.T
X = np.log1p(X)

# Align clinical properly
clinical_final = clinical_filtered[
    clinical_filtered["Patient_ID"].isin(common_patients)
].copy()

clinical_final = clinical_final.set_index("Patient_ID").loc[X.index]

y_survival = clinical_final["Survival_Days"]
y_virus = clinical_final["Virus_Status"]

print("Final shape:", X.shape)



Final shape: (229, 60660)


In [95]:
from sklearn.model_selection import train_test_split

print("Total samples:", len(X))

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_survival,
    test_size=0.2,
    random_state=42
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))


Total samples: 229
Train size: 183
Test size: 46


In [96]:
from sklearn.feature_selection import VarianceThreshold

# Remove genes with very low variance
selector = VarianceThreshold(threshold=0.01)
X_train_var = selector.fit_transform(X_train)
X_test_var = selector.transform(X_test)

print("After variance filter:", X_train_var.shape)


After variance filter: (183, 49455)


In [101]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_var)
X_test_scaled = scaler.transform(X_test_var)


In [100]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score

model = Ridge(alpha=1.0)
model.fit(X_train_scaled, y_train)

preds = model.predict(X_test_scaled)

print("MAE:", mean_absolute_error(y_test, preds))
print("R2:", r2_score(y_test, preds))


MAE: 634.071992730479
R2: -0.26591465612138854


In [102]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.01, max_iter=5000)
lasso.fit(X_train_scaled, y_train)

preds_lasso = lasso.predict(X_test_scaled)

print("MAE:", mean_absolute_error(y_test, preds_lasso))
print("R2:", r2_score(y_test, preds_lasso))


MAE: 719.5872804662521
R2: -0.9480062910260336


In [103]:
import numpy as np

nonzero = np.sum(lasso.coef_ != 0)
print("Selected genes:", nonzero)


Selected genes: 1167
